# GaLore Reproduction — Colab Runner

Before running anything: **Runtime > Change runtime type > T4 GPU**, then Save.

Each teammate should make their own copy of this notebook (**File > Save a copy in Drive**) and run it in their own session, so you can split the six configurations between you instead of waiting on one machine.

Assignments (adjust as your group prefers):
- Person A: `adamw`, `lora` rank 8
- Person B: `galore` rank 4, `galore` rank 16
- Person C: `galore` rank 64, `galore` rank 128

In [ ]:
!nvidia-smi

## 1. Upload the project files

Run the cell below, then in the dialog that appears select all three at once: `train.py`, `sanity_check.py`, and `requirements.txt`.

# New Section

In [ ]:
from google.colab import files
uploaded = files.upload()

## 2. Install dependencies

This takes a minute or two the first time.

In [6]:
!pip install -q -r requirements.txt

In [7]:
!pip install --upgrade datasets huggingface_hub fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 23.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 202

## 3. Sanity check

Confirms the model, data, and loss all behave before you spend GPU time on a full run. Look for `Sanity check passed` at the end.

In [3]:
!python sanity_check.py

Loading tokenizer and model...
Loading weights: 100% 197/197 [00:00<00:00, 9773.46it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading a small slice of SST-2...
README.md: 100% 35.3k/35.3k [00:00<00:00, 64.7MB/s]

sst2/train-00000-of-00001

## 4. Run your assigned configuration

Set `METHOD` and `RANK` below, run the cell, and wait for it to finish. Then change the values and run it again for each configuration you've been assigned.

- AdamW baseline: `METHOD = "adamw"`, `RANK = None`
- LoRA baseline: `METHOD = "lora"`, `RANK = 8`
- GaLore: `METHOD = "galore"`, `RANK = 4`, `16`, `64`, or `128`

In [ ]:
METHOD = "adamw"   # "adamw", "lora", or "galore"
RANK = None         # None for adamw, 8 for lora, or 4/16/64/128 for galore

if RANK is None:
    !python train.py --method {METHOD}
else:
    !python train.py --method {METHOD} --rank {RANK}

In [ ]:
import os
import time

# Complete matrix of all 6 reproduction configurations
experiments = [
    {"method": "adamw", "rank": None},
    {"method": "lora",   "rank": 8},
    {"method": "galore", "rank": 4},
    {"method": "galore", "rank": 16},
    {"method": "galore", "rank": 64},
    {"method": "galore", "rank": 128}
]

total_start = time.time()

for idx, exp in enumerate(experiments, 1):
    m = exp["method"]
    r = exp["rank"]

    print("\n" + "="*60)
    print(f"[{idx}/6] RUNNING: METHOD = {m.upper()} | RANK = {r}")
    print("="*60 + "\n")

    start_time = time.time()

    if r is None:
        !python train.py --method {m}
    else:
        !python train.py --method {m} --rank {r}

    elapsed = (time.time() - start_time) / 60
    print(f"\n--> Finished [{idx}/6] in {elapsed:.1f} minutes.\n")

total_elapsed = (time.time() - total_start) / 60
print(f"\nALL 6 EXPERIMENTS COMPLETED in {total_elapsed:.1f} minutes!")


[1/6] RUNNING: METHOD = ADAMW | RANK = None

=== Running configuration: adamw ===
Map: 100% 67349/67349 [00:02<00:00, 26182.21 examples/s]
Map: 100% 872/872 [00:00<00:00, 14005.91 examples/s]
Map: 100% 1821/1821 [00:00<00:00, 15773.82 examples/s]
Loading weights: 100% 197/197 [00:00<00:00, 4316.10it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

## 5. Check and download your results

Each run you complete adds one row here. Run this after every configuration to confirm it logged correctly.

In [ ]:
!cat results/results.csv

Once you've run every configuration assigned to you, download the results file and send it (or just the printed rows) to whoever is compiling the final README table.

In [ ]:
from google.colab import files
files.download('results/results.csv')

# Reham's working


In [ ]:
!nvidia-smi

Sat Sep 26 16:56:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
!pip install "huggingface_hub<0.25.0" "datasets>=2.20.0"

INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing install

In [2]:
!pip install --upgrade transformers huggingface_hub datasets peft accelerate

  Using cached huggingface_hub-2.0.0-py3-none-any.whl.metadata (16 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 80.3 MB/s eta 0:00:00
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 29.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.24.7
    Uninstalling huggingface-hub-0.24.7:
      Successfully uninstalled huggingface-hub-0.24.7
  Attempting uninstall: datasets
    Found existing installation: datasets 4.1.1
    Uninstalling datasets-4.1.1:
      Successfully uninstalled datasets-4.1.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.

In [2]:
!sed -i 's/"glue"/"nyu-mll\/glue"/g' sanity_check.py
!sed -i "s/'glue'/'nyu-mll\/glue'/g" sanity_check.py
!sed -i 's/"glue"/"nyu-mll\/glue"/g' train.py
!sed -i "s/'glue'/'nyu-mll\/glue'/g" train.py

verifying if train uses method /--rank flag or a .yaml config file


In [4]:
!python train.py --help


usage: train.py [-h] --method {adamw,lora,galore} [--rank RANK]
                [--epochs EPOCHS] [--batch_size BATCH_SIZE] [--lr LR]
                [--max_length MAX_LENGTH] [--output_dir OUTPUT_DIR]

GaLore reproduction: RoBERTa-base on SST-2

options:
  -h, --help            show this help message and exit
  --method {adamw,lora,galore}
  --rank RANK           Required for lora (use 8) and galore (use 4, 16, 64,
                        or 128)
  --epochs EPOCHS
  --batch_size BATCH_SIZE
  --lr LR
  --max_length MAX_LENGTH
  --output_dir OUTPUT_DIR
